# Partitionの変更

---

Slurm Partition の設定を変更します。デフォルト Partition の変更とアクセス制御（AllowGroups）の設定が可能です。

## 概要

このNotebookでは、既存のPartitionに対して以下の設定変更を行います：

1. **デフォルト Partition の変更**: ジョブ投入時に `-p` を省略した場合に使用される Partition の変更
2. **アクセス制御設定の変更**: Partition にジョブを投入できる UNIX グループの追加・削除

それぞれ独立したセクションになっています。必要なセクションのみ実行してください。

### 変更できないパラメータ

以下のパラメータはこのNotebookでは変更できません：

- **Nodesets**
    - Nodeset の所属変更（追加所属・所属解除）は「086-Nodesetの所属変更.ipynb」を使用してください
    - Feature 自体の作成・削除は 083/084 のnotebookを使用してください

### 前提条件

- 初期構築（010, 020）が完了していること
- 変更対象の Partition が存在すること

> VCノードの操作がないため、VCCアクセストークンは不要です。

## 準備

### UnitGroup名の指定

構築環境の UnitGroup名を指定します。

VCノードを作成時に指定した値を確認するために `group_vars` ファイル名の一覧を表示します。

In [ ]:
!ls -1 group_vars/*.yml | sed -e 's/^group_vars\///' -e 's/\.yml//' | sort

UnitGroup名を次のセルに指定してください。

In [ ]:
# (例)
# ugroup_name = 'OpenHPC'

ugroup_name = 

#### チェック

指定されたUnitGroup名が適切であることを確認します。

group_vars にfeature, partition 設定が記録されていることを確認します。

In [ ]:
%run scripts/group.py
gvars = load_group_vars(ugroup_name)

# group_vars にslurm_features, slurm_partitions が定義されていることを確認する
if not {"slurm_features", "slurm_partitions"} <= set(gvars.keys()):
    raise RuntimeError("Feature, Partition設定が未実施です")

Ansibleで接続できることを確認します。

In [ ]:
!ansible {ugroup_name}_master -m ping

### 現在のPartition 構成

現在の Partition 構成を表示します。

In [ ]:
%run scripts/group.py

gvars = load_group_vars(ugroup_name)

for pname, pconfig in gvars["slurm_partitions"].items():
    default_mark = " [DEFAULT]" if pconfig.get("default", False) else ""
    allow = pconfig.get("allow_groups", [])
    nodesets = pconfig["nodesets"]
    print(f"""
- {pname}{default_mark}
    Nodesets: {', '.join(nodesets)}
    AllowGroups: {format_allow_groups(allow)}""")

## デフォルトPartitionの変更（オプション）

デフォルト Partition を変更する場合はこのセクションを実行してください。
変更しない場合はこのセクションをスキップしてください。

デフォルト Partition は、ジョブ投入時に `-p` オプションを省略した場合に使用されます。
デフォルトは常に1つの Partition にのみ設定されます。
新たにデフォルトを指定すると、既存のデフォルト設定は自動的に解除されます。

現在のデフォルト Partition を確認します。

In [ ]:
current_default = get_default_partition(gvars)
print("Partition一覧:")
for pname in gvars["slurm_partitions"]:
    mark = " ← 現在のデフォルト" if pname == current_default else ""
    print(f"  - {pname}{mark}")

デフォルトにする Partition 名を指定してください。

In [ ]:
# (例)
# default_partition = 'compute'
# default_partition = 'gpu'

default_partition = 

## アクセス制御設定の変更（オプション）

Partition へのアクセスを特定の UNIX グループに制限する場合はこのセクションを実行してください。
変更しない場合はこのセクションをスキップしてください。

### 変更対象Partitionの指定

アクセス制御を変更する Partition 名を指定してください。

In [ ]:
# (例)
# allow_groups_partition = 'gpu'

allow_groups_partition = 

現在のアクセス制御設定を確認します。

In [ ]:
if allow_groups_partition not in gvars["slurm_partitions"]:
    msg = f"Partition '{allow_groups_partition}' が見つかりません。"
    raise ValueError(msg)

current_allow = gvars["slurm_partitions"][allow_groups_partition].get("allow_groups", [])
print(f"Partition '{allow_groups_partition}' の AllowGroups: {format_allow_groups(current_allow)}")

### アクセス制御グループの設定

新しい AllowGroups を指定してください。
制限なし（全ユーザー）に戻す場合は空リスト `[]` を指定してください。

In [ ]:
# (例)
# new_allow_groups = ["gpu-users", "admin"]  # グループ制限あり
# new_allow_groups = []                      # 制限なし（全ユーザー）

new_allow_groups = [
    
]

## 変更内容の確認

これまでに指定した変更内容をまとめて表示します。変更がない場合はその旨が表示されます。

In [ ]:
changes = []
apply_default = None
apply_allow_groups = None

if "default_partition" in vars() and default_partition:
    if default_partition not in gvars["slurm_partitions"]:
        msg = f"Partition '{default_partition}' が見つかりません。"
        raise ValueError(msg)
    current = get_default_partition(gvars)
    if current != default_partition:
        apply_default = default_partition
        changes.append(f"  Default: {current or '未設定'} → {default_partition}")

if "allow_groups_partition" in vars() and allow_groups_partition and "new_allow_groups" in vars():
    current_allow = gvars["slurm_partitions"][allow_groups_partition].get("allow_groups", [])
    new = new_allow_groups if new_allow_groups else []
    if set(current_allow) != set(new):
        apply_allow_groups = (allow_groups_partition, new)
        old_str = format_allow_groups(current_allow)
        new_str = format_allow_groups(new)
        changes.append(f"  AllowGroups ({allow_groups_partition}): {old_str} → {new_str}")

if changes:
    print("変更内容:")
    for c in changes:
        print(c)
else:
    print("変更はありません。")
    print("default_partition または new_allow_groups を指定してください。")

## 構成定義の更新

変更内容を `group_vars` に保存します。
この時点ではまだ Slurm クラスタには反映されていません。

In [ ]:
%run scripts/group.py

if not changes:
    print("変更がないためスキップします。")
else:
    if apply_default:
        set_default_partition(ugroup_name, apply_default)
        print(f"✓ デフォルト Partition を '{apply_default}' に変更しました")
    if apply_allow_groups:
        pname, groups = apply_allow_groups
        update_partition_allow_groups(ugroup_name, pname, groups)
        print(f"✓ AllowGroups ({pname}) を「{format_allow_groups(groups)}」に設定しました")

`group_vars` ファイルの内容を確認します。

In [ ]:
!cat group_vars/{ugroup_name}.yml

## Slurmへの反映

構成定義の変更をSlurmの設定ファイルに反映し、その再読み込みを行います。

ansible で `slurm.conf` を再生成し、Slurm に設定を再読み込みさせます。まずチェックモードで実行します。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v -CD playbooks/setup-slurm.yml

実際に変更を行います。

In [ ]:
!ansible-playbook -l {ugroup_name}_master -v playbooks/setup-slurm.yml

Slurm にPartition の設定が正しく反映されていることを確認します。

In [ ]:
!ansible {ugroup_name}_master -m shell \
    -a "scontrol show partition | grep -wE 'PartitionName|Default|AllowGroups'"